# 02 — TRACe Evaluator Validation (math half) — Group 16

**Goal:** prove our self-implemented TRACe metrics reproduce RAGBench's shipped *reference* scores, on every
domain, *before* we trust the evaluator on our own pipeline. (Mentor's golden rule: evaluation is the graded
core — validate it first.)

**What "validation" means here.** RAGBench ships ground-truth sentence labels (which doc sentences are
*relevant* `R`, which are *utilized* `U`, and which response sentences are *unsupported*) **and** the reference
TRACe scores GPT-4 derived from them. We recompute the scores from those gold labels with our own code and
compare:
- the 3 fraction metrics (relevance / utilization / completeness) → **RMSE** (typical gap; 0 = identical),
  worst single gap, and exact-match count;
- adherence (yes/no) → **accuracy**.

This validates the **arithmetic half only**. The hard half — an LLM judge that *produces* `R`/`U`/support for
our own pipeline's answers — is built and validated separately (notebook 03+).

> Runs on **CPU, no API key**. Imports the tested code in `src/evaluator/` rather than re-implementing it.

In [ ]:
# On Colab, clone the repo so `src/` is importable. Locally (running from the repo root), skip this.
import os, sys
if not os.path.exists("src"):
    !git clone https://github.com/abhisagar123/CapstoneRAG.git
    os.chdir("CapstoneRAG")
!pip install -q datasets
sys.path.insert(0, os.getcwd())

In [ ]:
from src.evaluator.trace import (
    total_doc_sentences, relevance, utilization, completeness, adherence,
    score_from_reference_labels,
)
from src.evaluator.validate import validate_all, DOMAIN_CONFIGS, FRACTION_METRICS
print("Domains:", list(DOMAIN_CONFIGS))

## 1. Sanity check on one example
The PubMedQA example we hand-worked: 18 sentences, |R|=11, |U|=9 → relevance 11/18=0.611, etc. Our functions
should match the shipped reference exactly.

In [ ]:
from datasets import load_dataset
ex = load_dataset("rungalileo/ragbench", "pubmedqa", split="test")[0]
ours = score_from_reference_labels(ex)
ref = {"relevance": ex["relevance_score"], "utilization": ex["utilization_score"],
       "completeness": ex["completeness_score"], "adherence": ex["adherence_score"]}
print(f'{"metric":13s} {"ours":>10s} {"reference":>10s}  match?')
for m in ["relevance","utilization","completeness","adherence"]:
    o, r = ours[m], ref[m]
    ok = "OK" if (o == r or (isinstance(o,float) and abs(o-r)<1e-9)) else "MISMATCH"
    print(f'{m:13s} {str(round(o,4) if isinstance(o,float) else o):>10s} {str(round(r,4) if isinstance(r,float) else r):>10s}  {ok}')

## 2. Full validation — all 12 configs, all 4 metrics
Up to 300 test rows per config. If the math half is correct, every RMSE is ~0 and adherence accuracy is 100%.

In [ ]:
import pandas as pd
reports = validate_all(n=300)
rows = []
for r in reports:
    row = {"domain": r["domain"], "config": r["config"], "n": r["n"]}
    for m in FRACTION_METRICS:
        row[f"{m}_rmse"] = r[m]["rmse"]
        row[f"{m}_exact"] = f'{r[m]["exact"]}/{r["n"]}'
    row["adherence_acc"] = r["adherence"]["accuracy"]
    rows.append(row)
df = pd.DataFrame(rows)
pd.set_option("display.float_format", lambda v: f"{v:.2e}")
df

In [ ]:
# Aggregate verdict.
tot = sum(r["n"] for r in reports)
worst = max(r[m]["rmse"] for r in reports for m in FRACTION_METRICS)
adh = sum(r["adherence"]["correct"] for r in reports)
print(f"Examples validated: {tot} across {len(reports)} configs")
print(f"Worst RMSE on any fraction metric, any domain: {worst:.2e}  (0 = perfect reproduction)")
print(f"Adherence accuracy: {adh}/{tot} = {adh/tot*100:.2f}%")
assert worst < 1e-6 and adh == tot, "validation regressed — investigate!"
print("\nPASS — math-half evaluator reproduces RAGBench reference scores on all domains.")

## 3. Notes & honest caveats
- **Sampled** up to 300 rows/config (~3.2k total), not the full test sets. Given exactly-zero error this is
  near-certain to generalise, but it is a sample.
- **Adherence** is `len(unsupported_response_sentence_keys) == 0`. We first tried
  `all(s['fully_supported'] ...)` and it matched only ~32% — `fully_supported` is `None` (blank) in most rows.
  Validation-at-scale caught a bug a single example had hidden. (See `AI_CONTEXT.md §9.6`.)
- **Relevance/utilization count keys with raw list length (no dedup)** to match how RAGBench computed them; a
  handful of CUAD rows have duplicate keys and only match under list semantics.
- This validates the **arithmetic** only. The LLM-judge half (producing labels for our own pipeline) is the
  real challenge and is validated separately.